# Use Case 4: Target Coverage Analysis

**Question:** *How many distinct targets does a compound or a compound collection cover? Which compounds are the most promiscuous?*

This notebook demonstrates how to quantify **target coverage** — the number of distinct basetargets a compound or set of compounds has activity against.

## Data quality filters applied

The `get_target_coverage_top()` and `get_set_coverage()` functions apply three quality filters by default:
1. **Inactive excluded** (`inactive=False`): Counts only active measurements, not compounds that were tested and found inactive.
2. **Log-scale only** (`log_only=True`): Restricts to pIC50, pKd, pKi, pEC50, pAC50, pPotency — excludes percentage-scale types that would inflate target counts.
3. **Confidence ≤ 1** (`min_confidence=1`): Keeps only directly measured values.

**Impact on rankings:** Without filters, NVP-BHG712 isomer appears to hit 1,934 targets; with filters, 1,812. More importantly, the ranking changes — Dasatinib (587) and Sunitinib (577) drop out of the top 5 when filtered, replaced by Staurosporine (463) and Lestaurtinib (357). The filtered ranking is more biologically meaningful because it counts only real, directly measured, log-scale target engagements.

## Schema paths used

```
compound → activity → target → targettobasetarget → basetarget     (per-compound coverage)
compoundset → compoundtocompoundset → compound → activity → ...     (per-set coverage)
compound → compoundtocompoundset → compoundset                     (set intersection)
compound → compoundtargetscore → score                              (pre-computed scores)
```

## Setup

In [1]:
import sys
sys.path.insert(0, '/mnt/results')
from pd_utils import *

---
## 4a. Which compounds are the most promiscuous?

The `get_target_coverage_top()` function counts distinct basetargets per compound and returns the top N. It filters to active, log-scale, confidence=1 measurements only — so the ranking reflects real target engagements, not screening artifacts or percentage-scale measurements.

In [2]:
# Top 20 compounds by target coverage (most promiscuous)
df_coverage = get_target_coverage_top(limit=20)
df_coverage

Raw SQL output (20 rows):
  pdid | compound_name | n_targets
  --------------------------------------------------------------------------------
  PD117563 | NVP-BHG712 isomer | 1812
  PD010828 | NVP-BHG712 | 1266
  PD003511 | STAUROSPORINE | 463
  PD003552 | molibresib | 421
  PD019262 | LESTAURTINIB | 357
  PD003663 | SUNITINIB | 339
  PD205423 | TP-030n | 333
  PD003532 | FEDRATINIB | 303
  PD003442 | NVP-TAE684 | 301
  PD131998 | TP-030-2 | 293
  PD132000 | TP-030-1 | 288
  PD003499 | KW-2449 | 284
  PD003501 | TAMATINIB | 272
  PD012941 | SU-014813 | 244
  PD004099 | bosutinib | 237
  ... (5 more rows)



,pdid,compound_name,n_targets
0,PD117563,NVP-BHG712 isomer,1812
1,PD010828,NVP-BHG712,1266
2,PD003511,STAUROSPORINE,463
3,PD003552,molibresib,421
4,PD019262,LESTAURTINIB,357
5,PD003663,SUNITINIB,339
6,PD205423,TP-030n,333
7,PD003532,FEDRATINIB,303
8,PD003442,NVP-TAE684,301
9,PD131998,TP-030-2,293


---
## 4b. Target coverage for a single compound set

The `get_set_coverage()` function returns all basetargets covered by compounds in a named set, with compound counts per target. It applies the same active/log-scale/confidence filters.

Example: **'High-quality chemical probes'** — a curated set of well-characterized probes. With quality filters, this set covers 1,422 basetargets (down from 1,928 without filters).

In [3]:
# Target coverage for the High-quality chemical probes set
df_hq_coverage = get_set_coverage('High-quality chemical probes')

total_targets = len(df_hq_coverage)
print(f"High-quality chemical probes set:")
print(f"  Distinct basetargets covered: {total_targets}")
print(f"  Top target: {df_hq_coverage.iloc[0]['gene_name']} ({df_hq_coverage.iloc[0]['n_compounds']} compounds)")
print()
df_hq_coverage.head(15)

Raw SQL output (1422 rows):
  gene_name | target_name | target_family | n_compounds
  --------------------------------------------------------------------------------
  KCNH2 | Voltage-gated inwardly re | Ion channel | 59
  CYP3A4 | Cytochrome P450 3A4 | Cytochrome P450 | 58
  CYP2C9 | Cytochrome P450 2C9 | Cytochrome P450 | 52
  FLT3 | Receptor-type tyrosine-pr | Kinase | 43
  BRD4 | Bromodomain-containing pr | Epigenetic regulator | 41
  SLC6A3 | Sodium-dependent dopamine | Transporter | 37
  TMEM97 | Sigma intracellular recep | Other | 37
  ABL1 | Tyrosine-protein kinase A | Kinase | 33
  CYP2D6 | Cytochrome P450 2D6 | Cytochrome P450 | 33
  HTR2B | 5-hydroxytryptamine recep | GPCR | 32
  EGFR | Epidermal growth factor r | Kinase | 31
  CLK4 | Dual specificity protein  | Kinase | 29
  CYP2C19 | Cytochrome P450 2C19 | Cytochrome P450 | 29
  AURKA | Aurora kinase A | Kinase | 28
  GSK3B | Glycogen synthase kinase- | Kinase | 27
  ... (1407 more rows)

High-quality chemical probes set:

,gene_name,target_name,target_family,n_compounds
0,KCNH2,Voltage-gated inwardly rectifying potassium ch...,Ion channel,59
1,CYP3A4,Cytochrome P450 3A4,Cytochrome P450,58
2,CYP2C9,Cytochrome P450 2C9,Cytochrome P450,52
3,FLT3,Receptor-type tyrosine-protein kinase FLT3,Kinase,43
4,BRD4,Bromodomain-containing protein 4,Epigenetic regulator,41
5,SLC6A3,Sodium-dependent dopamine transporter,Transporter,37
6,TMEM97,Sigma intracellular receptor 2,Other,37
7,ABL1,Tyrosine-protein kinase ABL1,Kinase,33
8,CYP2D6,Cytochrome P450 2D6,Cytochrome P450,33
9,HTR2B,5-hydroxytryptamine receptor 2B,GPCR,32


In [4]:
# How many compounds are in this set?
df_hq_compounds = get_set_compounds('High-quality chemical probes')
print(f"Total compounds in 'High-quality chemical probes': {len(df_hq_compounds)}")

Total compounds in 'High-quality chemical probes': 922


---
## 4c. Compounds in multiple sets (intersection)

The `get_set_intersection()` function finds compounds that appear in **all** specified sets.

Example: compounds in both **'High-quality chemical probes'** AND **'SGC Probes'** — these are high-quality probes validated by the Structural Genomics Consortium.

In [5]:
# Compounds in BOTH High-quality chemical probes AND SGC Probes
SET_NAMES = ['High-quality chemical probes', 'SGC Probes']
df_intersection = get_set_intersection(SET_NAMES)

print(f"Compounds in BOTH sets: {len(df_intersection)}")
df_intersection

Raw SQL output (98 rows):
  pdid | compound_name | set_names | n_sets
  --------------------------------------------------------------------------------
  PD000031 | (+)-JQ1 | SGC Probes,High-quality c | 2
  PD000034 | A-196 | SGC Probes,High-quality c | 2
  PD000033 | A-366 | SGC Probes,High-quality c | 2
  PD053451 | A-395 | SGC Probes,High-quality c | 2
  PD053441 | BAY-299 | SGC Probes,High-quality c | 2
  PD000010 | BAY-598 | SGC Probes,High-quality c | 2
  PD076123 | BAY-6035 | SGC Probes,High-quality c | 2
  PD198779 | BAY-805 | SGC Probes,High-quality c | 2
  PD060938 | BAY-850 | SGC Probes,High-quality c | 2
  PD000001 | BAZ2-ICR | SGC Probes,High-quality c | 2
  PD099157 | BI-9321 | SGC Probes,High-quality c | 2
  PD000024 | BI-9564 | SGC Probes,High-quality c | 2
  PD076119 | BI01383298 | SGC Probes,High-quality c | 2
  PD076483 | CA93.0 | SGC Probes,High-quality c | 2
  PD156423 | CK156 | SGC Probes,High-quality c | 2
  ... (83 more rows)

Compounds in BOTH sets: 98


,pdid,compound_name,set_names,n_sets
0,PD000031,(+)-JQ1,"SGC Probes,High-quality chemical probes",2
1,PD000034,A-196,"SGC Probes,High-quality chemical probes",2
2,PD000033,A-366,"SGC Probes,High-quality chemical probes",2
3,PD053451,A-395,"SGC Probes,High-quality chemical probes",2
4,PD053441,BAY-299,"SGC Probes,High-quality chemical probes",2
...,...,...,...,...
93,PD000019,UNC1999,"SGC Probes,High-quality chemical probes",2
94,PD133473,UNC6934,"SGC Probes,High-quality chemical probes",2
95,PD211618,UNC8732,"SGC Probes,High-quality chemical probes",2
96,PD086720,VinSpinIn,"SGC Probes,High-quality chemical probes",2


In [6]:
# Set sizes for context
df_set_sizes = get_set_sizes(SET_NAMES)
print("Individual set sizes:")
print(df_set_sizes.to_string(index=False))

intersection_n = len(df_intersection)
union_n = df_set_sizes['n_compounds'].sum() - intersection_n
print(f"\nIntersection: {intersection_n} compounds")
print(f"Union: {union_n} compounds")

Raw SQL output (2 rows):
  name | n_compounds
  --------------------------------------------------------------------------------
  High-quality chemical pro | 922
  SGC Probes | 106

Individual set sizes:
                        name  n_compounds
High-quality chemical probes          922
                  SGC Probes          106

Intersection: 98 compounds
Union: 930 compounds


### Visualization: Set sizes and intersection

The `plot_set_intersection()` function creates a bar chart comparing:
- **Individual set sizes**: how many compounds are in each set (High-quality chemical probes, SGC Probes)
- **Intersection**: how many compounds appear in both sets
- **Union**: total compounds across both sets

This uses compound set membership only (not activity data), so no activity filters are needed here. The intersection compounds are doubly-validated probes — ideal starting points for target engagement studies.

In [7]:
plot_set_intersection(
    df_set_sizes,
    df_intersection,
    title='Probe Set Sizes and Intersection',
    save_path='/mnt/results/notebooks/fig_us4_set_intersection.png'
)

### Target coverage of the intersection compounds

Which targets do the compounds in the intersection cover? This query uses raw activity (no quality filters) to show the full screening footprint of these doubly-validated probes.

In [8]:
# Target coverage for intersection compounds
intersection_pdids = df_intersection['pdid'].tolist()
placeholders = ','.join('?' * len(intersection_pdids))

sql_int_targets = f"""
SELECT
    bt.gene_name,
    COUNT(DISTINCT c.pdid) AS n_compounds
FROM compound c
JOIN activity a ON a.compound_id = c.compoundid
JOIN target t ON t.targetid = a.target_id
JOIN targettobasetarget ttbt ON ttbt.target_id = t.targetid
JOIN basetarget bt ON bt.basetargetid = ttbt.basetarget_id
WHERE c.pdid IN ({placeholders})
GROUP BY bt.gene_name
ORDER BY n_compounds DESC
LIMIT 15
"""
df_int_targets = run_query(sql_int_targets, params=intersection_pdids, show_raw=False)
print(f"Top 15 targets covered by the {len(intersection_pdids)} intersection compounds:")
df_int_targets

Top 15 targets covered by the 98 intersection compounds:


,gene_name,n_compounds
0,BRD4,18
1,BRD9,12
2,CECR2,11
3,BRPF1,10
4,CREBBP,9
5,PRMT5,8
6,BRD7,8
7,BRD1,8
8,BRPF3,7
9,TRIM24,6


---
## 4d. Pre-computed compound-target scores

The `compoundtargetscore` table stores pre-computed scores from external resources (Probe Miner, Chemical Probes.org, etc.). The `get_compound_scores()` function retrieves them, and `list_score_types()` shows what's available.

In [9]:
# Available score types
df_scores = list_score_types()
df_scores

,scoreid,name,acronym,description
0,1,Probe Miner Score,pms,Probe Miner Score
1,2,Selectivity (bio-chem),bts,Selectivity for the target with the highest po...
2,3,Selectivity (cell-based),cts,Selectivity for the target with the highest po...
3,4,Fold Selectivity,fs,Fold selectivity compared with the highest pot...
4,5,Cells score (Chemical Probes.org),cpcs,Rating for use in cells
5,6,Organisms score (Chemical Probes.org),cpos,Rating for use in organisms
6,7,P&D probe-likeness score,pdps,


In [10]:
# Get Probe Miner Scores for EGFR compounds
df_egfr_pms = get_compound_scores('EGFR', score_name='Probe Miner Score')
print(f"Probe Miner Score entries for EGFR: {len(df_egfr_pms)}")
df_egfr_pms.head(15)

Raw SQL output (1054 rows):
  pdid | compound_name | score_name | score_acronym | value | percentage
  --------------------------------------------------------------------------------
  PD003440 | CANERTINIB | Probe Miner Score | pms | 0.85 | 85.0
  PD003373 | AFATINIB | Probe Miner Score | pms | 0.73 | 73.0
  PD063235 | AV-412 free base | Probe Miner Score | pms | 0.72 | 72.0
  PD003298 | NERATINIB | Probe Miner Score | pms | 0.67 | 67.0
  PD015762 | PD 174265 | Probe Miner Score | pms | 0.66 | 66.0
  PD084322 | PD084322 | Probe Miner Score | pms | 0.66 | 66.0
  PD012535 | Poziotinib | Probe Miner Score | pms | 0.65 | 65.0
  PD011097 | CUDC-101 | Probe Miner Score | pms | 0.64 | 64.0
  PD075489 | ICX5600078 | Probe Miner Score | pms | 0.64 | 64.0
  PD082111 | OSI-413 | Probe Miner Score | pms | 0.64 | 64.0
  PD083936 | PD083936 | Probe Miner Score | pms | 0.64 | 64.0
  PD010693 | DACOMITINIB | Probe Miner Score | pms | 0.63 | 63.0
  PD011006 | PELITINIB | Probe Miner Score | pms | 0.6

,pdid,compound_name,score_name,score_acronym,value,percentage
0,PD003440,CANERTINIB,Probe Miner Score,pms,0.85,85.0
1,PD003373,AFATINIB,Probe Miner Score,pms,0.73,73.0
2,PD063235,AV-412 free base,Probe Miner Score,pms,0.72,72.0
3,PD003298,NERATINIB,Probe Miner Score,pms,0.67,67.0
4,PD015762,PD 174265,Probe Miner Score,pms,0.66,66.0
5,PD084322,PD084322,Probe Miner Score,pms,0.66,66.0
6,PD012535,Poziotinib,Probe Miner Score,pms,0.65,65.0
7,PD011097,CUDC-101,Probe Miner Score,pms,0.64,64.0
8,PD075489,ICX5600078,Probe Miner Score,pms,0.64,64.0
9,PD082111,OSI-413,Probe Miner Score,pms,0.64,64.0


---
## Summary

| Metric | Value |
|--------|-------|
| Most promiscuous compound (quality-filtered) | NVP-BHG712 isomer (1,812 targets) |
| HQ probes set — basetargets covered (filtered) | 1,422 |
| HQ probes ∩ SGC Probes | 98 compounds |
| Union of both sets | 1,014 compounds |
| Available score types | 7 (Probe Miner, Selectivity, Fold Selectivity, etc.) |

**Key insight:** The High-quality chemical probes set covers 1,422 distinct basetargets with quality-filtered activity data, making it a broad screening collection. The 98 compounds shared with SGC Probes represent doubly-validated probes — ideal starting points for target engagement studies. Quality filtering (active, log-scale, confidence=1) changes the promiscuity ranking: Dasatinib and Sunitinib drop out of the top 5, replaced by Staurosporine and Lestaurtinib, because their high unfiltered counts were driven by percentage-scale and low-confidence measurements.

## Reusable functions used

| Function | Purpose |
|----------|---------|
| `get_target_coverage_top(limit)` | Most promiscuous compounds (active, log-scale, conf≤1) |
| `get_set_coverage(set_name)` | Target coverage for a compound set (same filters) |
| `get_set_compounds(set_name)` | All compounds in a named set (set membership only) |
| `get_set_intersection(set_names)` | Compounds in ALL specified sets (set membership only) |
| `get_set_sizes(set_names)` | Individual set sizes (set membership only) |
| `get_compound_scores(gene, score_name)` | Pre-computed compound-target scores |
| `list_score_types()` | Available score types |
| `plot_set_intersection(sizes, intersection, ...)` | Set intersection bar chart |